# This is a simple code to plot functional maps calculated with fOptics software

In [ ]:
from utils import *

#analysis type
type_analysis = "Task_based"

#Path where data are saved (CHANGE IT)
# main_path ="/home/caredda/Videos/results_Functional_imaging"
main_path ="/home/caredda/Videos/test"

#Create output directory
if(os.path.isdir(main_path+"/results")):
    shutil.rmtree(main_path+"/results")

#Get patient directory
# array_path = glob.glob(main_path+"/*") # Load all patient directories
array_path = [main_path+"/P68"] # Load only one patient directory (CHANGE P68, according to the patient you want to analyze)

#Create output directory
os.mkdir(main_path+"/results")

os.mkdir(main_path+"/results/results_HbO2")
os.mkdir(main_path+"/results/results_Hb")

os.mkdir(main_path+"/results/results_HbO2/mask")
os.mkdir(main_path+"/results/results_Hb/mask")

os.mkdir(main_path+"/results/results_HbO2/Z_stats")
os.mkdir(main_path+"/results/results_Hb/Z_stats")


# Manual thresholding of statistical maps: TO BE ADJUSTED
Thresh = 1.15

#Resize factor (1 = no resize)
f = 1

#Loop over patients
for i in range(len(array_path)):

    print(array_path[i])

    #Get list of different analyses for "type_analysis"
    analysis_path = glob.glob(array_path[i]+"/"+type_analysis+"*")

    #get id patient
    id_Patient = os.path.basename(array_path[i])

    #Load reference image
    ref_img = cv2.imread(array_path[i]+"/ref.png")

    #Get reference surgical window contour
    ref_mask = (cv2.imread(array_path[i]+"/ref_mask.png",cv2.IMREAD_GRAYSCALE)/255).astype(np.uint8)

    for _analysis_path in analysis_path:

        #Get analysis path
        analysis_name = os.path.basename(_analysis_path)
        path = _analysis_path+"/SPM_MBLL"

        #Get functional images
        out_HbO2, mask, SPM_mask_HbO2, auto_mask_HbO2, Z_stats_HbO2 = get_functional_img_task_based(path, 0, Thresh, f, ref_img, ref_mask)
        out_Hb, mask, SPM_mask_Hb, auto_mask_Hb, Z_stats_Hb  = get_functional_img_task_based(path,1,Thresh,f, ref_img, ref_mask)

        #Write stats masks
        cv2.imwrite(main_path+"/results/results_HbO2/mask/SPM_"+id_Patient+".png",SPM_mask_HbO2)
        cv2.imwrite(main_path+"/results/results_Hb/mask/SPM_"+id_Patient+".png",SPM_mask_Hb)
        cv2.imwrite(main_path+"/results/results_HbO2/mask/Auto_"+id_Patient+".png",auto_mask_HbO2)
        cv2.imwrite(main_path+"/results/results_Hb/mask/Auto_"+id_Patient+".png",auto_mask_Hb)

        #Write z-stats images
        np.savetxt(main_path+"/results/results_HbO2/Z_stats/"+id_Patient+".txt",Z_stats_HbO2)
        np.savetxt(main_path+"/results/results_Hb/Z_stats/"+id_Patient+".txt",Z_stats_Hb)



        if(not os.path.isdir(path+"/results")):
            os.mkdir(path+"/results")

        #Write image with statistical inferences
        cv2.imwrite(main_path+"/results/results_HbO2/"+id_Patient+"_"+analysis_name+".png",out_HbO2)
        cv2.imwrite(main_path+"/results/results_Hb/"+id_Patient+"_"+analysis_name+".png",out_Hb)

        